# Week 9 — GaMMA to NonLinLoc conversion and absolute location
Converts GaMMA's event/assignment output into NonLinLoc's .obs format (no
built-in helper like PyOcto's to_nonlinloc(), so built directly against the
same format confirmed in Week 6), then reuses Week 6's existing travel-time
grids directly - they don't depend on which picker/associator produced the
picks, only the station geometry and velocity model, both unchanged.

In [ ]:
import os
import pandas as pd
from obspy import UTCDateTime

PROJECT_DIR = os.path.expanduser("~/tvz_project")
CATALOG_DIR = os.path.join(PROJECT_DIR, "catalogs")
NLL_DIR = os.path.join(PROJECT_DIR, "nonlinloc")
NLL_PN_DIR = os.path.join(NLL_DIR, "phasenet")
os.makedirs(os.path.join(NLL_PN_DIR, "obs"), exist_ok=True)
os.makedirs(os.path.join(NLL_PN_DIR, "loc"), exist_ok=True)

events = pd.read_csv(os.path.join(CATALOG_DIR, "phasenet_gamma_events.csv"))
assignments = pd.read_csv(os.path.join(CATALOG_DIR, "phasenet_gamma_assignments.csv"))
declustered = pd.read_csv(os.path.join(PROJECT_DIR, "picks", "phasenet", "phasenet_declustered.csv"))

print(f"{len(events)} events, {len(assignments)} assignments, {len(declustered)} declustered picks")

In [ ]:
# Merge assignments -> declustered picks by position (pick_index is the row
# position in the pick_df passed into association(), which was built directly
# from declustered with a clean 0..N-1 index - confirmed by the sanity check
# in the next cell, not just assumed).
declustered_indexed = declustered.reset_index(drop=True)
merged = assignments.merge(
    declustered_indexed, left_on="pick_index", right_index=True, how="left"
)
print(f"Merged: {len(merged)} rows")
print(merged[["event_index", "station", "time", "phase", "probability"]].head(10))

In [ ]:
# Sanity check on the pick_index alignment: picks assigned to the same event
# should cluster tightly in time (a few seconds to tens of seconds apart at
# most for a local earthquake) - if the indexing were wrong, picks within an
# "event" would be scattered randomly across the whole 61-day window instead.
merged["time_dt"] = pd.to_datetime(merged["time"])
check = merged.groupby("event_index")["time_dt"].agg(["min", "max"])
check["span_seconds"] = (check["max"] - check["min"]).dt.total_seconds()
print("Time span of picks within each event (should be small - seconds to ~1 minute):")
print(check["span_seconds"].describe())
print(f"\nEvents with a span > 120s (possible indexing problem if this is common): "
      f"{(check['span_seconds'] > 120).sum()} of {len(check)}")

In [ ]:
# Build the .obs file - same NLLOC_OBS format used for EQTransformer in Week 6
obs_path = os.path.join(NLL_PN_DIR, "obs", "phasenet_gamma.obs")

with open(obs_path, "w") as f:
    for event_idx, group in merged.groupby("event_index"):
        f.write(f"PUBLIC_ID E{int(event_idx):08d}\n")
        for _, row in group.iterrows():
            t = UTCDateTime(row["time"])
            phase = row["phase"].upper()
            date_str = t.strftime("%Y%m%d")
            hm_str = t.strftime("%H%M")
            sec_str = f"{t.second + t.microsecond / 1e6:7.4f}"
            f.write(
                f"{row['station']} ?    ?    ? {phase}      ? {date_str} {hm_str} "
                f"{sec_str} GAU  5.00e-02 -1.00e+00 -1.00e+00 -1.00e+00\n"
            )
        f.write("\n")

print(f"Saved: {obs_path}")
with open(obs_path) as f:
    print("".join(f.readlines()[:8]))

In [ ]:
# Build GTSRCE lines the same way as Week 6, using the exact station labels
# that appear in our own picks (guarantees a match against the .obs file).
import obspy

INVENTORY_FILE = os.path.join(PROJECT_DIR, "data", "inventory", "NZ_2019-08-01_2019-09-30.xml")
inventory = obspy.read_inventory(INVENTORY_FILE)

station_coords = {}
for net in inventory:
    for sta in net:
        key = f"{net.code}.{sta.code}"
        station_coords[key] = (sta.latitude, sta.longitude, sta.elevation / 1000.0)

unique_labels = sorted(merged["station"].unique())
print(f"{len(unique_labels)} unique station labels in the GaMMA-associated picks")

gtsrce_lines = []
missing = []
for label in unique_labels:
    parts = label.split(".")
    lookup_key = f"{parts[0]}.{parts[1]}"
    if lookup_key not in station_coords:
        missing.append(label)
        continue
    lat, lon, elev_km = station_coords[lookup_key]
    gtsrce_lines.append(f"GTSRCE {label} LATLON {lat:.5f} {lon:.5f} 0.0 {elev_km:.4f}")

if missing:
    print(f"WARNING: {len(missing)} labels had no inventory match: {missing}")
else:
    print("All station labels matched to inventory coordinates")
print(f"{len(gtsrce_lines)} GTSRCE lines built")

In [ ]:
# Build the NLLoc control file. Reuses Week 6's existing travel-time grid
# directly (time_grid_root) - the grid depends only on station geometry and
# the velocity model, both unchanged, not on which picker/associator
# produced the events being located.
TARGET_LAT, TARGET_LON = -38.8, 176.0
time_grid_root = os.path.join(NLL_DIR, "time", "tvz_layer")
output_root = os.path.join(NLL_PN_DIR, "loc", "phasenet")

nlloc_lines = [
    "CONTROL 1 54321",
    f"TRANS SIMPLE {TARGET_LAT} {TARGET_LON} 0.0",
    "",
    "LOCSIG TVZ earthquake catalogue project - UEA MSc dissertation (PhaseNet/GaMMA)",
    "LOCCOM PhaseNet picks, GaMMA association, NonLinLoc EDT_OT_WT location",
    f"LOCFILES {obs_path} NLLOC_OBS {time_grid_root} {output_root}",
    "LOCHYPOUT SAVE_NLLOC_ALL FILENAME_PUBLIC_ID",
    "LOCSEARCH OCT 20 20 10 0.01 20000 5000 0 1",
    "LOCGRID 301 301 44 -150.0 -150.0 -3.0 1.0 1.0 1.0 PROB_DENSITY SAVE",
    "LOCMETH EDT_OT_WT 9999.0 4 -1 -1 -1 -1 -1 1",
    "LOCGAU 0.2 0.0",
    "LOCPHASEID P P",
    "LOCPHASEID S S",
    "LOCQUAL2ERR 0.1 0.5 1.0 2.0 99999.9",
    "LOCANGLES ANGLES_NO 5",
] + gtsrce_lines

nlloc_path = os.path.join(NLL_PN_DIR, "nlloc_phasenet.in")
with open(nlloc_path, "w") as f:
    f.write("\n".join(nlloc_lines) + "\n")

print(f"Saved: {nlloc_path}")
print("\n".join(nlloc_lines[:12]))

In [ ]:
# Apply the same quality filters used for the EQTransformer catalogue
# (Week 6/7), reusing the identical .hyp parser and thresholds.
import glob

LOC_DIR = os.path.join(NLL_PN_DIR, "loc")
hyp_files = sorted(glob.glob(os.path.join(LOC_DIR, "*.grid0.loc.hyp")))
print(f"{len(hyp_files)} .hyp files found")

def parse_hyp(path):
    result = {"file": os.path.basename(path)}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            if parts[0] == "GEOGRAPHIC":
                result["origin_time"] = f"{parts[2]}-{parts[3]}-{parts[4]}T{parts[5]}:{parts[6]}:{parts[7]}"
                result["lat"] = float(parts[9])
                result["lon"] = float(parts[11])
                result["depth_km"] = float(parts[13])
            elif parts[0] == "QUALITY":
                result["rms"] = float(parts[8])
                result["nphs"] = float(parts[10])
                result["gap"] = float(parts[12])
            elif parts[0] == "STATISTICS":
                cov_xx = float(parts[8])
                cov_yy = float(parts[14])
                result["horiz_error_km"] = (cov_xx + cov_yy) ** 0.5
    return result

records = [parse_hyp(f) for f in hyp_files]
catalog = pd.DataFrame(records)
print(f"Parsed {len(catalog)} events")

filtered_pn = catalog[
    (catalog["horiz_error_km"] < 5.0) &
    (catalog["gap"] < 180.0) &
    (catalog["nphs"] >= 8) &
    (catalog["rms"] < 0.5)
].copy()

print(f"\nEvents retained: {len(filtered_pn)} of {len(catalog)} "
      f"({100 * len(filtered_pn) / len(catalog):.1f}%)")
print("\nRejection breakdown:")
print(f"  horiz_error >= 5km: {(catalog['horiz_error_km'] >= 5.0).sum()}")
print(f"  gap >= 180deg:      {(catalog['gap'] >= 180.0).sum()}")
print(f"  nphs < 8:           {(catalog['nphs'] < 8).sum()}")
print(f"  rms >= 0.5s:        {(catalog['rms'] >= 0.5).sum()}")

raw_out = os.path.join(CATALOG_DIR, "phasenet_nlloc_raw.csv")
filtered_out = os.path.join(CATALOG_DIR, "phasenet_nlloc_quality_filtered.csv")
catalog.to_csv(raw_out, index=False)
filtered_pn.to_csv(filtered_out, index=False)
print(f"\nSaved: {raw_out}")
print(f"Saved: {filtered_out}")